# Notebook Colab (T4) — RAG Formulaire

Ce notebook prépare un environnement Colab T4 pour tester le pipeline RAG sur les formulaires IRCC en français. Il permet de :

- Vérifier le GPU disponible et configurer le dépôt.
- Installer les dépendances et construire un petit index.
- Poser des questions sans passer par la CLI interactive.

> **Astuce :** utilisez un quota réduit de formulaires (ex. 30) pour accélérer l'ingestion sur Colab.

> **Note :** Ce notebook utilise maintenant le code intégré directement depuis le dépôt. Les patches précédents ont été intégrés dans les modules source.

## 1) Vérifier le GPU

In [1]:
!nvidia-smi

Tue Dec  2 00:45:16 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2) Préparer le dépôt

- Définissez `RAG_FORM_REPO_URL` si le dépôt n'est pas déjà présent dans `/content/rag-formulaire`.
- Le notebook ajoute automatiquement le dépôt au `PYTHONPATH` pour l'installation en mode développement.

In [2]:
import os
import pathlib
import sys

REPO_URL = os.environ.get("RAG_FORM_REPO_URL", "").strip()
REPO_URL = "https://github.com/abdelmajidlra/rag-formulaire.git"
WORKDIR = pathlib.Path("/content/rag-formulaire")

if not WORKDIR.exists():
    if not REPO_URL:
        raise ValueError(
            "Définissez RAG_FORM_REPO_URL ou clonez le dépôt dans /content/rag-formulaire avant d'exécuter ce notebook."
        )
    else:
        print(f"Clonage du dépôt depuis {REPO_URL}…")
        get_ipython().system(f"git clone {REPO_URL} {WORKDIR}")

get_ipython().run_line_magic("cd", str(WORKDIR))
if str(WORKDIR) not in sys.path:
    sys.path.append(str(WORKDIR))

# Add the 'src' directory to sys.path for direct module imports
SRC_DIR = WORKDIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

Clonage du dépôt depuis https://github.com/abdelmajidlra/rag-formulaire.git…
Cloning into '/content/rag-formulaire'...
remote: Enumerating objects: 187, done.
remote: Counting objects: 100% (187/187), done.
remote: Compressing objects: 100% (127/127), done.
remote: Total 187 (delta 94), reused 92 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (187/187), 89.45 KiB | 1.33 MiB/s, done.
Resolving deltas: 100% (94/94), done.
/content/rag-formulaire


## 3) Installer les dépendances

L'installation en mode développement (`-e .`) permet de modifier le code localement pendant la session Colab.

In [3]:
get_ipython().system("pip -q install -U pip setuptools wheel")
get_ipython().system("pip -q install -e .")
!pip install -q \
    requests beautifulsoup4 tqdm langdetect pydantic rank-bm25 \
    chromadb sentence-transformers scikit-learn transformers torch \
    click docling pdfplumber \
    pypdf pymupdf \
    bitsandbytes accelerate
!pip install -q pikepdf



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 81.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... don

## 4) Paramétrage rapide

Vous pouvez ajuster les variables pour contrôler la taille de l'ingestion et activer/désactiver GraphRAG.
- `RAG_FORM_MIN_FORMS`: nombre minimum de formulaires (les formulaires synthétiques complètent si besoin).
- `RAG_FORM_MAX_SYNTH`: nombre maximal de formulaires synthétiques générés.
- `RAG_FORM_BASE_DIR`: dossier racine où les données (`data/`) seront écrites.

In [4]:
from pprint import pprint

os.environ.setdefault("RAG_FORM_BASE_DIR", str(WORKDIR))
os.environ.setdefault("RAG_FORM_MIN_FORMS", "30")
os.environ.setdefault("RAG_FORM_MAX_SYNTH", "0")
os.environ.setdefault("RAG_FORM_ENABLE_GRAPHRAG", "false")

print("Configuration en cours :")
pprint({k: os.environ[k] for k in sorted(os.environ) if k.startswith("RAG_FORM_")})

Configuration en cours :
{'RAG_FORM_BASE_DIR': '/content/rag-formulaire',
 'RAG_FORM_ENABLE_GRAPHRAG': 'false',
 'RAG_FORM_MAX_SYNTH': '0',
 'RAG_FORM_MIN_FORMS': '30'}


## 5) Construire l'index (BM25 + vecteur)

Cette étape télécharge les formulaires, découpe les documents puis construit les index. Ajustez `min_forms` pour accélérer sur Colab.

> **Optimisations intégrées:**
> - LLM Singleton: Une seule instance du modèle est chargée (économise ~50% de mémoire)
> - Downloader Deduplication: Évite les doublons dans le manifest
> - Smart Retrieval: Détection automatique des codes de formulaire spécifiques

### Aperçu du manifest

In [5]:
%%writefile complete_reindex.py
#!/usr/bin/env python3
"""
Complete re-indexing workflow with enhanced validation and error recovery.
FIXED VERSION: Removed invalid imports and corrected index filenames.
"""

import logging
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path(__file__).parent / "src"))

from rag_formulaire import config
# Import correct de la pipeline principale
from rag_formulaire.ingest import ingest_pipeline

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


def cleanup_corrupt_pdfs():
    """Remove any corrupt PDFs from previous downloads."""
    logger.info("Step 1/3: Cleaning up corrupt PDFs...")

    if not config.RAW_FORMS_DIR.exists():
        logger.info("No existing PDFs to clean up")
        return 0

    corrupt_count = 0
    for pdf_file in config.RAW_FORMS_DIR.glob("*.pdf"):
        try:
            with open(pdf_file, 'rb') as f:
                header = f.read(1024).lower()

            is_corrupt = (
                not header.startswith(b'%pdf-') or
                b'<!doctype' in header or
                b'<html' in header or
                pdf_file.stat().st_size < 5120
            )

            if is_corrupt:
                logger.warning(f"Removing corrupt PDF: {pdf_file.name}")
                pdf_file.unlink()
                corrupt_count += 1
        except Exception as e:
            logger.error(f"Error checking {pdf_file.name}: {e}")

    logger.info(f"Removed {corrupt_count} corrupt PDFs")
    return corrupt_count


def run_ingestion():
    """Run the full ingestion pipeline."""
    logger.info("Step 2/3: Running full ingestion pipeline (Download -> Parse -> Index)...")
    try:
        # ingest_pipeline handles downloading, parsing (with XFA fix), and indexing
        ingest_pipeline()
        logger.info("✓ Ingestion pipeline completed")
    except Exception as e:
        logger.error(f"Ingestion failed: {e}")
        raise


def validate_index():
    """Validate the index quality."""
    logger.info("Step 3/3: Validating index quality...")

    # Check that index directories exist and have content
    # CORRECTION: bm25.pkl est le bon nom de fichier (pas index.pkl)
    checks = {
        "BM25 index": config.BM25_DIR / "bm25.pkl",
        "Vector index": config.CHROMA_DIR,
        "Chunks file": config.CHUNKS_PATH,
        "Manifest": config.MANIFEST_PATH,
    }

    all_ok = True
    for name, path in checks.items():
        if path.exists():
            if path.is_file():
                size = path.stat().st_size
                logger.info(f"✓ {name}: {size:,} bytes")
            else:
                logger.info(f"✓ {name}: directory exists")
        else:
            logger.error(f"✗ {name}: NOT FOUND at {path}")
            all_ok = False

    return all_ok


def main():
    """Execute complete re-indexing workflow."""
    logger.info("="*70)
    logger.info("RAG FORMULAIRE - COMPLETE RE-INDEXING (FIXED)")
    logger.info("="*70)

    try:
        # Step 1: Cleanup
        cleanup_corrupt_pdfs()

        # Step 2: Full Ingestion (Download, Parse, Index)
        run_ingestion()

        # Step 3: Validate
        if validate_index():
            logger.info("")
            logger.info("="*70)
            logger.info("✓ RE-INDEXING COMPLETED SUCCESSFULLY")
            logger.info("="*70)
        else:
            logger.error("Index validation failed!")
            sys.exit(1)

    except Exception as e:
        logger.error(f"Re-indexing failed: {e}", exc_info=True)
        sys.exit(1)


if __name__ == "__main__":
    main()

Overwriting complete_reindex.py


In [6]:
# Correction complète : nettoyage des PDF corrompus et ré-indexation stricte
!python complete_reindex.py

INFO:numexpr.utils:NumExpr defaulting to 2 threads.
2025-12-02 00:46:44.608983: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764636404.639601     866 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764636404.649065     866 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764636404.671847     866 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764636404.671878     866 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764636404.671888     8

In [7]:
import json
from rag_formulaire import config

with open(config.MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)

print(f"Formulaires disponibles : {len(manifest)}")
for entry in manifest[:3]:
    print(entry)

Formulaires disponibles : 30
{'form_code': 'IMM 0008', 'title_fr': 'Annexe 9\xa0: Immigration économique – Déclaration d’intention de résider au Québec', 'pdf_url': 'https://www.canada.ca/content/dam/ircc/migration/ircc/francais/pdf/trousses/form/imm0008_9f.pdf', 'local_path': '/content/rag-formulaire/data/raw/forms/IMM_0008.pdf', 'category': None, 'last_updated': None}
{'form_code': 'IMM 0016', 'title_fr': 'Déclaration solennelle pour le parent d’un mineur aux fins de l’entrée au Canada pour les membres de la famille élargie décrets concernant la COVID-19 pris en vertu de la loi sur la mise en quarantaine', 'pdf_url': 'https://www.canada.ca/content/dam/ircc/documents/pdf/francais/trousses/form/imm0016f.pdf', 'local_path': '/content/rag-formulaire/data/raw/forms/IMM_0016.pdf', 'category': None, 'last_updated': None}
{'form_code': 'IMM 5741', 'title_fr': 'Renvoi des frais de traitement ou des frais relatifs au droit de résidence permanente', 'pdf_url': 'https://www.canada.ca/content/dam

## 6) Poser des questions (sans CLI)

Le bloc suivant instancie les composants du pipeline et expose une fonction `ask_question` pour tester rapidement vos requêtes en français.

In [8]:
from rag_formulaire import config
from rag_formulaire.evaluation import AdvancedSelfReflector, CRAGEvaluator, verify_response_against_evidence
from rag_formulaire.indexing import load_indexes
from rag_formulaire.llm import LocalLLM
from rag_formulaire.query_processing import AgenticQueryRouter, MultilingualQueryHandler, QueryDecomposer, QueryExpander
from rag_formulaire.reranker import CrossEncoderReranker
from rag_formulaire.retrieval import HybridRetriever

config.CHUNK_SIZE = 200
config.CHUNK_OVERLAP = 30  # Petit chevauchement pour garder le contexte

index_store = load_indexes()
query_handler = MultilingualQueryHandler()
router = AgenticQueryRouter()
expander = QueryExpander()
decomposer = QueryDecomposer()
retriever = HybridRetriever(index_store)
reranker = CrossEncoderReranker()
evaluator = CRAGEvaluator()
reflector = AdvancedSelfReflector()
llm = LocalLLM()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [9]:
def ask_question(question: str, evidence_k: int = config.FINAL_EVIDENCE_K):
    q_orig, q_fr = query_handler.normalize(question)
    route = router.route(q_fr)
    expansions = expander.expand(q_fr, n=3)
    subqueries = decomposer.decompose(q_fr) if route == "MULTI_STEP" else [q_fr]

    candidates = []
    for sub in subqueries:
        for variant in expansions:
            candidates.extend(retriever.retrieve(variant, manifest=None))

    reranked = reranker.rerank(q_fr, candidates, top_n=config.RERANK_TOP_N)
    if not reranked:
        return {"route": route, "answer": "Aucun extrait trouvé.", "evidence": []}

    scores = list(range(len(reranked), 0, -1))
    if not evaluator.is_evidence_strong(scores, reranked):
        return {"route": route, "answer": evaluator.fallback_message(), "evidence": []}

    evidence_texts = [
        f"[{c.base_chunk.form_code}] {c.base_chunk.section_title}: {c.base_chunk.content}"
        for c in reranked[:evidence_k]
    ]
    system_prompt = (
        "Vous êtes un assistant spécialisé dans les formulaires IRCC. Répondez uniquement en français en vous basant sur les "
        "extraits fournis. Citez le code du formulaire et la section."
    )
    user_prompt = q_fr + "Extraits:" + "".join(evidence_texts)
    answer = llm.chat(system_prompt, user_prompt, max_new_tokens=256)

    if verify_response_against_evidence(answer, reranked):
        answer = reflector.reflect(q_fr, answer, reranked)
    else:
        answer = evaluator.fallback_message()

    return {
        "route": route,
        "expansions": expansions,
        "answer": answer,
        "evidence": reranked[:evidence_k],
    }

### Utilitaires d'affichage

Fonction pour afficher les résultats de manière formatée avec Markdown.

In [10]:
from IPython.display import display, Markdown

def display_result(result, manifest_list=None):
    """
    Affiche la réponse et les sources de manière formatée en Markdown.
    """
    # 1. En-tête avec la route utilisée
    md = f"### 🤖 Réponse (Stratégie : `{result['route']}`)\n\n"

    # 2. La réponse générée
    md += f"{result['answer']}\n\n"

    # 3. Les sources (Preuves)
    md += "---\n#### 🔍 Sources utilisées :\n"

    # Création d'un dictionnaire pour retrouver les URL à partir du code formulaire
    url_map = {m['form_code']: m['pdf_url'] for m in manifest_list} if manifest_list else {}

    for i, ev in enumerate(result['evidence'], 1):
        chunk = ev.base_chunk
        form_code = chunk.form_code
        section = chunk.section_title

        # Lien vers le PDF officiel si disponible
        if form_code in url_map:
            source_link = f"[{form_code}]({url_map[form_code]})"
        else:
            source_link = f"**{form_code}**"

        # Petit extrait du texte pour contexte (nettoyé des sauts de ligne)
        preview = chunk.content.replace("\n", " ")[:500] + "..."

        md += f"{i}. {source_link} — *{section}* (Page {chunk.page_number})\n"
        md += f"   > <small>{preview}</small>\n"

    display(Markdown(md))

### Gestion de la mémoire GPU

Outils pour nettoyer le cache GPU entre les requêtes et éviter les erreurs OOM (Out Of Memory).

In [11]:
import torch
import gc

# 1. Force Python garbage collection
gc.collect()

# 2. Clear CUDA (GPU) cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✅ GPU cache cleared.")

# 3. Check status
!nvidia-smi

✅ GPU cache cleared.
Tue Dec  2 01:06:34 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P0             28W /   70W |    7008MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------------

## 7) Tests avec questions variées

Testez le pipeline avec une série de questions pour valider le système.

> **Note:** Le système détectera automatiquement les codes de formulaire spécifiques (comme "IMM 5476") et filtrera les résultats en conséquence.

In [12]:
import torch
import gc
import os
import logging

# --- CORRECTIF : Cacher les avertissements de génération ---
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()  # Bloque les warnings "generation flags"
# -----------------------------------------------------------

# On garde un contexte court pour la mémoire
os.environ["RAG_FORM_FINAL_EVIDENCE_K"] = "3"

test_questions = [
    # --- ✅ PARTIE 1 : Questions sur des documents INDEXÉS (Doivent réussir) ---
    "Qui doit signer le formulaire IMM 5476 pour désigner un représentant ?",
    "Que doit-on déclarer à propos des maladies mentales dans le questionnaire médical IMM 5955 ?",
    "Quels documents peuvent servir de preuve d'expérience de travail au Canada selon le formulaire IMM 0134 ?",
    "À qui s'adresse l'offre d'emploi pour les ressortissants étrangers dispensés d'EIMT (IMM 0116) ?",
    "Quel est le rôle de l'interprète décrit dans le formulaire IMM 5744 ?",
    "Quelles informations l'employeur doit-il fournir sur l'adresse commerciale dans le formulaire IMM 0267 ?",
    "Quelles sont les responsabilités de l'employeur concernant l'offre d'emploi dans le formulaire IMM 0273 ?",
    "Dans quel cas les frais relatifs au droit de résidence permanente sont-ils remboursés (IMM 5741) ?",

    # --- ❌ PARTIE 2 : Questions "Test de Sécurité" (Documents ABSENTS) ---
    "Quels sont les documents requis dans la liste de contrôle IMM 5488 ?",
    "Qui doit être listé dans le formulaire de renseignements sur la famille IMM 5707 ?"
]

def force_cleanup():
    """Nettoyage mémoire agressif"""
    # Supprimer explicitement les variables globales si elles existent
    if 'result' in globals(): del globals()['result']
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Décorateur pour désactiver le calcul de gradient (économie VRAM massive)
@torch.no_grad()
def traiter_une_question(index, total, question_text):
    print(f"▶️ Question {index}/{total}: {question_text}")

    # Détection si c'est une question piège (juste pour l'affichage ici)
    est_piege = "5488" in question_text or "5707" in question_text

    try:
        if 'ask_question' not in globals():
            print("❌ Erreur: Fonction ask_question manquante.")
            return

        # Appel du RAG
        result = ask_question(question_text)

        # Analyse de la réponse
        answer = result.get('answer', '')
        evidence = result.get('evidence', [])

        # Affichage du résultat
        if evidence:
            sources = list(set([c.base_chunk.form_code for c in evidence]))
            print(f"   ✅ Sources trouvées : {sources}")
            print(f"   🤖 Réponse : {answer[:300]}...")
        else:
            print("   ⚠️ Aucun extrait trouvé (Comportement normal pour documents absents).")
            if est_piege:
                print("   🛡️  SUCCÈS DU TEST DE SÉCURITÉ : Le modèle n'a pas halluciné !")
            else:
                print(f"   🤖 Réponse de repli : {answer}")

        # Suppression explicite
        del result
        del answer
        del evidence

    except Exception as e:
        print(f"   ⚠️ Erreur : {e}")

# --- Lancement ---
print(f"🚀 Lancement du test hybride ({len(test_questions)} questions)...\n")

# Premier nettoyage avant de commencer
force_cleanup()

for i, question in enumerate(test_questions, 1):
    traiter_une_question(i, len(test_questions), question)
    print("-" * 50)
    force_cleanup() # Nettoyage après chaque tour

print("✅ Série terminée.")

🚀 Lancement du test hybride (10 questions)...

▶️ Question 1/10: Qui doit signer le formulaire IMM 5476 pour désigner un représentant ?


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


   ✅ Sources trouvées : ['IMM 5476']
   🤖 Réponse : Le representant devra signer la partie "Déclaration de votre représentant" de la forme IMM 5476 pour être considéré comme le représentant autorisé du demandeur auprès d'Immigration, Réfugiés et Citoyenneté Canada et de l'Agence des services frontaliers du Canada. La signature et la date doivent être...
--------------------------------------------------
▶️ Question 2/10: Que doit-on déclarer à propos des maladies mentales dans le questionnaire médical IMM 5955 ?
   ✅ Sources trouvées : ['IMM 5955']
   🤖 Réponse : Vous devez déclarer toute maladie mentale ou condition médicale grave lorsque vous remplissez le questionnaire médical IMM 5955. Bien qu'il n'y ait pas spécifiquement une section consacrée aux maladies mentales dans cette partie du formulaire, il est demandé d'en fournir des informations supplémenta...
--------------------------------------------------
▶️ Question 3/10: Quels documents peuvent servir de preuve d'expérience de 